# ENBIOS4TIMES: Environmental Assessment of Energy System Scenarios with ENBIOS

This notebook guides TIMES model users through the process of assessing the environmental impacts of energy system scenarios using ENBIOS2 (Environmental and Bioeconomic System Analysis).

ENBIOS2 integrates **Life Cycle Assessment (LCA)** with the **MuSIASEM** (Multi-Scale Integrated Analysis of Societal and Ecosystem Metabolism) framework to evaluate energy system pathways produced by models such as TIMES.

**Workflow:**
1. What is ENBIOS?
2. Building the Node Hierarchy (JSON dendrogram)
3. Creating an ENBIOS Experiment
4. The ENBIOS–TIMES Activity Dictionary
5. Running the Environmental Assessment

## 1. What is ENBIOS?

ENBIOS2 is a Python-based simulation tool for the environmental and bioeconomic assessment of energy system pathways. It is designed to work with the outputs of Energy System Optimisation Models (ESOMs) such as TIMES, Calliope, or OSeMOSYS.

ENBIOS2 does not perform LCA calculations itself — it uses [Brightway2](https://docs.brightway.dev/) as its LCA engine, connected to life cycle inventory databases such as [ecoinvent](https://ecoinvent.org/).

### Key concepts

| Concept | Description |
|---------|-------------|
| **Experiment** | The top-level object that holds the hierarchy, adapters, methods and scenarios |
| **Hierarchy (dendrogram)** | A tree structure of nodes representing the energy system |
| **Structural node (leaf)** | An energy technology linked to a Brightway/ecoinvent activity |
| **Functional node** | An aggregation node (e.g. wind energy, total electricity) |
| **Adapter** | Connects ENBIOS to an LCA database (e.g. Brightway adapter for ecoinvent) |
| **Scenario** | A set of activity outputs representing one energy system pathway from TIMES |
| **Method** | An LCIA method (e.g. ReCiPe GWP1000) used to calculate environmental impacts |

### How TIMES connects to ENBIOS

TIMES produces scenario results as **energy flows** (e.g. PJ of electricity from wind, solar, gas). Each TIMES technology is mapped to an ecoinvent activity via an **activity dictionary**. ENBIOS then calculates the environmental impacts of each technology and aggregates them through the hierarchy.

### Setup and imports

In [ ]:
import bw2data
import pandas as pd
from pathlib import Path

from enbios import Experiment, report
from enbios.base.models import ExperimentData

# Check available Brightway projects and databases
report()

In [ ]:
# Set your Brightway project and ecoinvent database
PROJECT_NAME = "ecoinvent_391"   # replace with your project name
DATABASE = "ecoinvent_391_cutoff"  # replace with your database name

bw2data.projects.set_current(PROJECT_NAME)
db = bw2data.Database(DATABASE)

## 2. Building the Node Hierarchy (JSON Dendrogram)

The hierarchy defines the structure of your energy system as a tree. It mirrors your TIMES model structure:
- **Root node**: the whole energy system
- **Functional nodes**: technology groups (e.g. wind, solar, gas)
- **Structural nodes (leaves)**: individual technologies linked to ecoinvent activities

Each leaf node must specify:
- `name`: a label for the technology
- `adapter`: which LCA adapter to use (use `"brightway-adapter"` for ecoinvent)
- `config`: the ecoinvent activity identifier (`code`, or `name` + `location` + `unit`)

### Example hierarchy for a simplified electricity system

In [ ]:
# Example hierarchy — replace activity codes with your TIMES technologies
# To find ecoinvent activity codes: db.search("technology name", filter={"location": "ES"})

hierarchy = {
    "name": "root",
    "aggregator": "sum-aggregator",
    "children": [
        {
            "name": "wind",
            "aggregator": "sum-aggregator",
            "children": [
                {
                    "name": "wind_onshore",
                    "adapter": "brightway-adapter",
                    "config": {
                        "code": "ed3da88fc23311ee183e9ffd376de89b",  # wind onshore ES
                        "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
                    }
                },
                {
                    "name": "wind_offshore",
                    "adapter": "brightway-adapter",
                    "config": {
                        "code": "6ebfe52dc3ef5b4d35bb603b03559023",  # wind offshore ES
                        "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
                    }
                }
            ]
        },
        {
            "name": "solar",
            "aggregator": "sum-aggregator",
            "children": [
                {
                    "name": "solar_tower",
                    "adapter": "brightway-adapter",
                    "config": {
                        "code": "f2700b2ffcb6b32143a6f95d9cca1721",
                        "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
                    }
                },
                {
                    "name": "solar_parabolic",
                    "adapter": "brightway-adapter",
                    "config": {
                        "code": "19040cdacdbf038e2f6ad59814f7a9ed",
                        "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
                    }
                }
            ]
        }
    ]
}

print("Hierarchy defined with", len(hierarchy["children"]), "technology groups")

## 3. Creating an ENBIOS Experiment

An ENBIOS experiment combines:
- The **hierarchy** (node tree)
- The **adapter** configuration (Brightway project + LCIA methods)
- The **scenarios** (one per TIMES energy pathway)

### LCIA methods
ENBIOS uses Brightway LCIA methods. The example below uses ReCiPe 2016 midpoint methods. Replace with the methods relevant to your assessment.

In [ ]:
# Define the LCIA methods — replace with your chosen impact categories
experiment_methods = {
    "GWP1000": (
        "ReCiPe 2016 v1.03, midpoint (H)",
        "climate change",
        "global warming potential (GWP1000)",
    ),
    "WCP": (
        "ReCiPe 2016 v1.03, midpoint (E)",
        "water use",
        "water consumption potential (WCP)",
    ),
    "LOP": (
        "ReCiPe 2016 v1.03, midpoint (E)",
        "land use",
        "agricultural land occupation (LOP)",
    ),
}

# Assemble the full experiment configuration
experiment_config = {
    "adapters": [
        {
            "adapter_name": "brightway-adapter",
            "config": {"bw_project": PROJECT_NAME},
            "methods": experiment_methods,
        }
    ],
    "hierarchy": hierarchy,
}

# Validate the configuration before running
exp_data = ExperimentData(**experiment_config)
print("Configuration valid")
exp_data.model_dump(exclude_unset=True)

## 4. The ENBIOS–TIMES Activity Dictionary

The **activity dictionary** maps each TIMES technology to an ecoinvent activity. This is the most important step for TIMES users — it defines the correspondence between your model's technologies and the LCA database.

### Structure
Each entry maps a TIMES technology name to:
- The ecoinvent activity `code` (preferred — unique identifier)
- OR `name` + `location` + `unit` (if code is unknown)
- The `unit` and `magnitude` of the default output (to match TIMES energy units)

### Finding ecoinvent codes
Use Brightway's search to find the right activity:

In [ ]:
# Search for an ecoinvent activity by name and location
# Replace with your TIMES technology name
results = db.search("electricity production, wind, 1-3MW turbine, onshore", 
                    filter={"location": "ES"})
for r in results:
    print(r["name"], "|", r["location"], "|", r["code"])

In [ ]:
# ENBIOS-TIMES Activity Dictionary
# Format: TIMES_technology_name -> ecoinvent activity config
# Replace these entries with your own TIMES technologies

TIMES_ACTIVITY_DICT = {
    # TIMES technology name   : ecoinvent config
    "EWIN1N": {  # example TIMES code for onshore wind
        "adapter": "brightway-adapter",
        "config": {
            "code": "ed3da88fc23311ee183e9ffd376de89b",
            "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
        }
    },
    "EWIN2N": {  # example TIMES code for offshore wind
        "adapter": "brightway-adapter",
        "config": {
            "code": "6ebfe52dc3ef5b4d35bb603b03559023",
            "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
        }
    },
    "ESOL1N": {  # example TIMES code for solar tower
        "adapter": "brightway-adapter",
        "config": {
            "code": "f2700b2ffcb6b32143a6f95d9cca1721",
            "default_output": {"unit": "kilowatt_hour", "magnitude": 1}
        }
    },
}

print(f"Activity dictionary contains {len(TIMES_ACTIVITY_DICT)} technologies")

## 5. Running the Socio-Ecological Assessment

Each TIMES scenario is a set of energy outputs per technology (e.g. kWh produced by each plant in a given year). These are passed to ENBIOS as **scenario nodes**.

### Unit conversion
TIMES outputs are typically in **PJ** (petajoules). Convert to kWh before passing to ENBIOS:
- 1 PJ = 277,777,778 kWh

### Running and reading results

In [ ]:
# Define TIMES scenarios as ENBIOS scenario nodes
# Each scenario corresponds to one TIMES run / year / pathway
# Units must match the default_output units defined in the hierarchy

PJ_TO_KWH = 277_777_778  # conversion factor

scenarios = [
    {
        "name": "TIMES_scenario_2030",
        "nodes": {
            "wind_onshore":   {"unit": "kilowatt_hour", "magnitude": 4 * PJ_TO_KWH},
            "wind_offshore":  {"unit": "kilowatt_hour", "magnitude": 3 * PJ_TO_KWH},
            "solar_tower":    {"unit": "kilowatt_hour", "magnitude": 3 * PJ_TO_KWH},
            "solar_parabolic":{"unit": "kilowatt_hour", "magnitude": 4 * PJ_TO_KWH},
        }
    },
    {
        "name": "TIMES_scenario_2050",
        "nodes": {
            "wind_onshore":   {"unit": "kilowatt_hour", "magnitude": 8 * PJ_TO_KWH},
            "wind_offshore":  {"unit": "kilowatt_hour", "magnitude": 6 * PJ_TO_KWH},
            "solar_tower":    {"unit": "kilowatt_hour", "magnitude": 5 * PJ_TO_KWH},
            "solar_parabolic":{"unit": "kilowatt_hour", "magnitude": 7 * PJ_TO_KWH},
        }
    }
]

# Add scenarios to the experiment config
experiment_config["scenarios"] = scenarios

# Create and run the experiment
experiment = Experiment(experiment_config)
print(experiment.info())

In [ ]:
# Run all scenarios
results = experiment.run()
results

In [ ]:
# Export results to CSV for further analysis
output_path = Path("results/ENBIOS_TIMES_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

experiment.results_to_csv(str(output_path))
df = pd.read_csv(output_path).fillna("")
df